In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Composite DNA Decoder: Training & Evaluation - Eta-Based Variable Ratio
# ## Extended alphabet with variable mixture ratios (η=0.2, ℓ∈{-2,-1,0,1,2})

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "2"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
ERROR_MODEL = "grass"

# ------------------- ETA-BASED ALPHABET PARAMETERS -------------------
ETA = 0.2
ELL_VALUES = [-2, -1, 0, 1, 2]

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 50

# Calculate vocab size
NUM_PURE_BASES = 4
NUM_TWO_MIX_PAIRS = 6
VOCAB_SIZE = NUM_PURE_BASES + NUM_TWO_MIX_PAIRS * len(ELL_VALUES)  # 34

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    
    # Eta Parameters
    "eta": ETA,
    "ell_values": ELL_VALUES,
    "alphabet_mode": f"eta{ETA}",
    
    # Data Paths
    "dataset_dir": "./dataset",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Results directory
    "results_dir": f"./results_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_eta{ETA}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZE,
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50],
    
    # Model Architecture
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Eta: {CONFIG['eta']}, Ell Values: {CONFIG['ell_values']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Theoretical Capacity: {np.log2(CONFIG['vocab_size']):.4f} bits/position")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 5: BUILD ETA-BASED ALPHABET MAPPINGS
# =============================================================================

def build_eta_based_symbol_to_idx(eta, ell_values):
    """Build symbol-to-index mapping for eta-based alphabet."""
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    pair_names = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']
    current_idx = 4
    
    for pair_name in pair_names:
        for ell in ell_values:
            if ell >= 0:
                symbol_name = f"{pair_name}_ell{ell}"
            else:
                symbol_name = f"{pair_name}_ell_neg{abs(ell)}"
            symbol_to_idx[symbol_name] = current_idx
            current_idx += 1
    
    return symbol_to_idx


def build_eta_based_ideal_vectors(eta, ell_values):
    """
    Build ideal frequency vectors for eta-based alphabet.
    
    Returns:
        torch.Tensor of shape (vocab_size, 4)
    """
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
    ]
    
    # Two-mix pairs: (vec_idx1, vec_idx2)
    pair_indices = [
        (0, 1),  # B1: A|C
        (0, 2),  # B2: A|G
        (0, 3),  # B3: A|T
        (1, 2),  # B4: C|G
        (1, 3),  # B5: C|T
        (2, 3),  # B6: G|T
    ]
    
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            prob1 = 0.5 + ell * eta
            prob2 = 0.5 - ell * eta
            
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
    
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_eta_based_symbol_to_idx(CONFIG["eta"], CONFIG["ell_values"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_eta_based_ideal_vectors(CONFIG["eta"], CONFIG["ell_values"]).to(device)

print(f"\n📊 Symbol Mappings (η={CONFIG['eta']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")
print(f"\n   Sample mappings (first 10):")
for i, (sym, idx) in enumerate(sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1])[:10]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<16} {idx:<4} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}, {vec[3]:.2f}]")
print(f"   ... ({len(SYMBOL_TO_IDX) - 10} more)")


# =============================================================================
# CELL 6: DATA PREPROCESSING (Same as previous)
# =============================================================================
# def preprocess_cluster_to_matrix(cluster_reads, target_length): ...
# Copy from previous code

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


# =============================================================================
# CELL 7: PYTORCH DATASET CLASS (Same structure, updated for eta)
# =============================================================================

class CompositeDNADatasetEta(Dataset):
    """PyTorch Dataset for Eta-Based Composite DNA data."""
    
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


# =============================================================================
# CELL 8: NEURAL NETWORK MODEL (Same as previous)
# =============================================================================
# class CompositeDecoderLSTM(nn.Module): ...
# Copy from previous code

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        # x: (Batch, 4, L) -> (Batch, L, 4)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        # Return: (Batch, vocab_size, L)
        return logits.permute(0, 2, 1)


# =============================================================================
# CELL 9: BASELINE DECODERS (Same logic, works with any ideal_vectors)
# =============================================================================
# def min_distance_decoder(obs, ideal_vectors): ...
# def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01): ...
# def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01): ...
# Copy from previous code

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder (L2 norm)."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 10: EARLY STOPPING CLASS (Same as previous)
# =============================================================================
# class EarlyStopping: ...
# Copy from previous code

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss


# =============================================================================
# CELL 11: TRAINING FUNCTION (Same as previous)
# =============================================================================
# def train_model(model, train_loader, val_loader, config, weights_path, device): ...
# Copy from previous code

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """Train the model with warmup + cosine annealing scheduler."""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=config['epochs'] - config['warmup_epochs'], eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # Training
        model.train()
        train_loss_accum = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item()
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # Validation
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        avg_val_loss = val_loss_accum / len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
        
        scheduler.step()
        early_stopper(avg_val_loss, model)
        
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history


# =============================================================================
# CELL 12: EVALUATION FUNCTION (Same as previous)
# =============================================================================
# def evaluate_all_decoders(model, loader, ideal_vectors, device): ...
# Copy from previous code

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """Evaluate all 4 decoders on the given data loader."""
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)
            
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies


# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """Run complete experiment for a single coverage level."""
    
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Eta: {config['eta']}, Vocab Size: {config['vocab_size']}")
    print(f"{'='*70}")
    
    set_seed(config['seed'])
    
    full_ds = CompositeDNADatasetEta(
        config['dataset_path'], config['seq_length'], symbol_to_idx, limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    model_prefix = f"{config['error_name']}_eta{config['eta']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} (η={config['eta']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history


# =============================================================================
# CELL 14: PLOTTING FUNCTIONS (Same as previous)
# =============================================================================
# def plot_training_history(history, coverage_M, save_path, config): ...
# def plot_comparison_results(results, save_path, config): ...
# Copy from previous code

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training & Validation Loss (M={coverage_M}, η={config["eta"]})', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'Learning Rate Schedule (M={coverage_M})', fontsize=14)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: η={config['eta']} ({config['vocab_size']} classes)\n"
             f"Error Model: {config['error_name']}, Seq Length: {config['seq_length']}")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")


# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset generator first with:\n"
        f"   ETA = {CONFIG['eta']}, ELL_VALUES = {CONFIG['ell_values']}"
    )

with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Vocab Size: {data['metadata']['vocab_size']}")
print(f"   Eta: {data['metadata']['eta']}")


# =============================================================================
# CELL 16: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'seq_length': CONFIG['seq_length'],
        'eta': CONFIG['eta'],
        'ell_values': CONFIG['ell_values'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    plot_prefix = f"{CONFIG['error_name']}_eta{CONFIG['eta']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)


# =============================================================================
# CELL 17: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

print(f"\n   Eta: {CONFIG['eta']}, Vocab Size: {CONFIG['vocab_size']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Eta: {CONFIG['eta']}, Classes: {CONFIG['vocab_size']}")
print(f"   Results directory: {CONFIG['results_dir']}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 CONFIGURATION
   Error Model: grass (G15)
   Eta: 0.2, Ell Values: [-2, -1, 0, 1, 2]
   Sequence Length: 104
   Vocab Size: 34 classes
   Theoretical Capacity: 5.0875 bits/position
   Dataset Path: ./dataset/dna_G15_eta0.2_100000_50.pkl
   Results Dir: ./results_G15_eta0.2
🎲 Random seed set to: 42

📊 Symbol Mappings (η=0.2):
   Total symbols: 34

   Sample mappings (first 10):
   A                0    [1.00, 0.00, 0.00, 0.00]
   C                1    [0.00, 1.00, 0.00, 0.00]
   G                2    [0.00, 0.00, 1.00, 0.00]
   T                3    [0.00, 0.00, 0.00, 1.00]
   B1_ell_neg2      4    [0.10, 0.90, 0.00, 0.00]
   B1_ell_neg1      5    [0.30, 0.70, 0.00, 0.00]
   B1_ell0          6    [0.50, 0.50, 0.00, 0.00]
   B1_ell1          7    [0.70, 0.30, 0.00, 0.00]
   B1_ell2          8    [0.90, 0.10, 0.00, 0.00]
   B2_ell_neg2      9    [0.10, 0.00, 0.90, 0.00]
   ... (24 more)

📦 LOADING DATASET
✅ Dataset loaded: ./dataset/d

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 2.8732 | Val: 2.8747 | LR: 1.00e-03 | Time: 30.1s
      ✓ Val loss improved (2.8764 → 2.8747). Saving...
   Epoch 012/100 | Train: 2.8720 | Val: 2.8743 | LR: 1.00e-03 | Time: 29.4s
      ✓ Val loss improved (2.8747 → 2.8743). Saving...
   Epoch 013/100 | Train: 2.8712 | Val: 2.8738 | LR: 9.99e-04 | Time: 30.8s
      ✓ Val loss improved (2.8743 → 2.8738). Saving...
   Epoch 014/100 | Train: 2.8704 | Val: 2.8726 | LR: 9.97e-04 | Time: 30.0s
      ✓ Val loss improved (2.8738 → 2.8726). Saving...
   Epoch 015/100 | Train: 2.8700 | Val: 2.8728 | LR: 9.95e-04 | Time: 30.3s
      EarlyStopping counter: 1/10
   Epoch 016/100 | Train: 2.8694 | Val: 2.8726 | LR: 9.92e-04 | Time: 32.4s
      ✓ Val loss improved (2.8726 → 2.8726). Saving...
   Epoch 017/100 | Train: 2.8691 | Val: 2.8723 | LR: 9.89e-04 | Time: 30.2s
      ✓ Val loss improved (2.8726 → 2.8723). Saving...
   Epoch 018/100 | Train: 2.8687 | Val: 2.8719 | LR: 9.85e-04 | Time: 30.0s
      ✓ Val loss improved (2

   Epoch 078/100 | Train: 2.8367 | Val: 2.8444 | LR: 1.54e-04 | Time: 31.0s
      ✓ Val loss improved (2.8445 → 2.8444). Saving...
   Epoch 079/100 | Train: 2.8366 | Val: 2.8444 | LR: 1.41e-04 | Time: 30.2s
      ✓ Val loss improved (2.8444 → 2.8444). Saving...
   Epoch 080/100 | Train: 2.8366 | Val: 2.8445 | LR: 1.29e-04 | Time: 30.2s
      EarlyStopping counter: 1/10
   Epoch 081/100 | Train: 2.8364 | Val: 2.8447 | LR: 1.18e-04 | Time: 30.1s
      EarlyStopping counter: 2/10
   Epoch 082/100 | Train: 2.8362 | Val: 2.8444 | LR: 1.07e-04 | Time: 31.1s
      EarlyStopping counter: 3/10
   Epoch 083/100 | Train: 2.8361 | Val: 2.8445 | LR: 9.64e-05 | Time: 30.1s
      EarlyStopping counter: 4/10
   Epoch 084/100 | Train: 2.8360 | Val: 2.8446 | LR: 8.64e-05 | Time: 29.9s
      EarlyStopping counter: 5/10
   Epoch 085/100 | Train: 2.8359 | Val: 2.8443 | LR: 7.69e-05 | Time: 29.9s
      ✓ Val loss improved (2.8444 → 2.8443). Saving...
   Epoch 086/100 | Train: 2.8359 | Val: 2.8444 | LR: 6.79

   Epoch 044/100 | Train: 2.3554 | Val: 2.3681 | LR: 7.04e-04 | Time: 52.5s
      EarlyStopping counter: 1/10
   Epoch 045/100 | Train: 2.3549 | Val: 2.3628 | LR: 6.88e-04 | Time: 54.9s
      ✓ Val loss improved (2.3630 → 2.3628). Saving...
   Epoch 046/100 | Train: 2.3551 | Val: 2.3621 | LR: 6.71e-04 | Time: 51.8s
      ✓ Val loss improved (2.3628 → 2.3621). Saving...
   Epoch 047/100 | Train: 2.3533 | Val: 2.3623 | LR: 6.55e-04 | Time: 56.5s
      EarlyStopping counter: 1/10
   Epoch 048/100 | Train: 2.3527 | Val: 2.3620 | LR: 6.38e-04 | Time: 50.3s
      ✓ Val loss improved (2.3621 → 2.3620). Saving...
   Epoch 049/100 | Train: 2.3523 | Val: 2.3617 | LR: 6.21e-04 | Time: 50.8s
      ✓ Val loss improved (2.3620 → 2.3617). Saving...
   Epoch 050/100 | Train: 2.3510 | Val: 2.3626 | LR: 6.04e-04 | Time: 52.2s
      EarlyStopping counter: 1/10
   Epoch 051/100 | Train: 2.3508 | Val: 2.3619 | LR: 5.87e-04 | Time: 50.6s
      EarlyStopping counter: 2/10
   Epoch 052/100 | Train: 2.3510 | V

...

   Epoch 020/100 | Train: 0.7010 | Val: 0.6780 | LR: 9.76e-04 | Time: 658.1s
      ✓ Val loss improved (0.6801 → 0.6780). Saving...
   Epoch 021/100 | Train: 0.6974 | Val: 0.6755 | LR: 9.70e-04 | Time: 616.9s
      ✓ Val loss improved (0.6780 → 0.6755). Saving...
   Epoch 022/100 | Train: 0.6947 | Val: 0.6729 | LR: 9.64e-04 | Time: 617.2s
      ✓ Val loss improved (0.6755 → 0.6729). Saving...
   Epoch 023/100 | Train: 0.6919 | Val: 0.6712 | LR: 9.57e-04 | Time: 623.7s
      ✓ Val loss improved (0.6729 → 0.6712). Saving...
   Epoch 024/100 | Train: 0.6890 | Val: 0.6692 | LR: 9.49e-04 | Time: 627.3s
      ✓ Val loss improved (0.6712 → 0.6692). Saving...
   Epoch 025/100 | Train: 0.6867 | Val: 0.6694 | LR: 9.42e-04 | Time: 632.2s
      EarlyStopping counter: 1/10
   Epoch 026/100 | Train: 0.6845 | Val: 0.6663 | LR: 9.33e-04 | Time: 609.8s
      ✓ Val loss improved (0.6692 → 0.6663). Saving...
   Epoch 027/100 | Train: 0.6823 | Val: 0.6650 | LR: 9.24e-04 | Time: 625.1s
      ✓ Val loss imp

   Epoch 085/100 | Train: 0.6448 | Val: 0.6380 | LR: 7.69e-05 | Time: 612.6s
      ✓ Val loss improved (0.6382 → 0.6380). Saving...
   Epoch 086/100 | Train: 0.6447 | Val: 0.6378 | LR: 6.79e-05 | Time: 614.2s
      ✓ Val loss improved (0.6380 → 0.6378). Saving...
   Epoch 087/100 | Train: 0.6446 | Val: 0.6381 | LR: 5.95e-05 | Time: 621.8s
      EarlyStopping counter: 1/10
   Epoch 088/100 | Train: 0.6445 | Val: 0.6380 | LR: 5.16e-05 | Time: 615.4s
      EarlyStopping counter: 2/10
   Epoch 089/100 | Train: 0.6444 | Val: 0.6377 | LR: 4.42e-05 | Time: 606.6s
      ✓ Val loss improved (0.6378 → 0.6377). Saving...
   Epoch 090/100 | Train: 0.6444 | Val: 0.6376 | LR: 3.74e-05 | Time: 602.8s
      ✓ Val loss improved (0.6377 → 0.6376). Saving...
   Epoch 091/100 | Train: 0.6444 | Val: 0.6375 | LR: 3.11e-05 | Time: 610.4s
      ✓ Val loss improved (0.6376 → 0.6375). Saving...
   Epoch 092/100 | Train: 0.6441 | Val: 0.6376 | LR: 2.54e-05 | Time: 665.0s
      EarlyStopping counter: 1/10
   Epoc

   Epoch 043/100 | Train: 0.5102 | Val: 0.4956 | LR: 7.19e-04 | Time: 807.1s
      ✓ Val loss improved (0.4960 → 0.4956). Saving...
   Epoch 044/100 | Train: 0.5094 | Val: 0.4956 | LR: 7.04e-04 | Time: 839.9s
      ✓ Val loss improved (0.4956 → 0.4956). Saving...
   Epoch 045/100 | Train: 0.5086 | Val: 0.4959 | LR: 6.88e-04 | Time: 828.0s
      EarlyStopping counter: 1/10
   Epoch 046/100 | Train: 0.5077 | Val: 0.4938 | LR: 6.71e-04 | Time: 850.5s
      ✓ Val loss improved (0.4956 → 0.4938). Saving...
   Epoch 047/100 | Train: 0.5070 | Val: 0.4929 | LR: 6.55e-04 | Time: 819.0s
      ✓ Val loss improved (0.4938 → 0.4929). Saving...
   Epoch 048/100 | Train: 0.5060 | Val: 0.4935 | LR: 6.38e-04 | Time: 861.0s
      EarlyStopping counter: 1/10
   Epoch 049/100 | Train: 0.5056 | Val: 0.4925 | LR: 6.21e-04 | Time: 840.8s
      ✓ Val loss improved (0.4929 → 0.4925). Saving...
   Epoch 050/100 | Train: 0.5050 | Val: 0.4918 | LR: 6.04e-04 | Time: 832.0s
      ✓ Val loss improved (0.4925 → 0.491

   Epoch 004/100 | Train: 0.9491 | Val: 0.7887 | LR: 3.70e-04 | Time: 1035.0s
      ✓ Val loss improved (1.0224 → 0.7887). Saving...
   Epoch 005/100 | Train: 0.7798 | Val: 0.6702 | LR: 4.60e-04 | Time: 1018.8s
      ✓ Val loss improved (0.7887 → 0.6702). Saving...
   Epoch 006/100 | Train: 0.6988 | Val: 0.6075 | LR: 5.50e-04 | Time: 1049.9s
      ✓ Val loss improved (0.6702 → 0.6075). Saving...
   Epoch 007/100 | Train: 0.6502 | Val: 0.5755 | LR: 6.40e-04 | Time: 1040.0s
      ✓ Val loss improved (0.6075 → 0.5755). Saving...
   Epoch 008/100 | Train: 0.6155 | Val: 0.5367 | LR: 7.30e-04 | Time: 1030.8s
      ✓ Val loss improved (0.5755 → 0.5367). Saving...
   Epoch 009/100 | Train: 0.5848 | Val: 0.5172 | LR: 8.20e-04 | Time: 1020.8s
      ✓ Val loss improved (0.5367 → 0.5172). Saving...
   Epoch 010/100 | Train: 0.5604 | Val: 0.4913 | LR: 9.10e-04 | Time: 1019.2s
      ✓ Val loss improved (0.5172 → 0.4913). Saving...
   Epoch 011/100 | Train: 0.5366 | Val: 0.4758 | LR: 1.00e-03 | Time:

   Epoch 068/100 | Train: 0.3868 | Val: 0.3768 | LR: 2.97e-04 | Time: 1054.3s
      EarlyStopping counter: 1/10
   Epoch 069/100 | Train: 0.3864 | Val: 0.3766 | LR: 2.82e-04 | Time: 1019.7s
      ✓ Val loss improved (0.3766 → 0.3766). Saving...
   Epoch 070/100 | Train: 0.3860 | Val: 0.3763 | LR: 2.66e-04 | Time: 1012.8s
      ✓ Val loss improved (0.3766 → 0.3763). Saving...
   Epoch 071/100 | Train: 0.3859 | Val: 0.3759 | LR: 2.51e-04 | Time: 1021.3s
      ✓ Val loss improved (0.3763 → 0.3759). Saving...
   Epoch 072/100 | Train: 0.3855 | Val: 0.3753 | LR: 2.36e-04 | Time: 1012.6s
      ✓ Val loss improved (0.3759 → 0.3753). Saving...
   Epoch 073/100 | Train: 0.3853 | Val: 0.3753 | LR: 2.21e-04 | Time: 1011.9s
      ✓ Val loss improved (0.3753 → 0.3753). Saving...
   Epoch 074/100 | Train: 0.3850 | Val: 0.3749 | LR: 2.07e-04 | Time: 1012.1s
      ✓ Val loss improved (0.3753 → 0.3749). Saving...
   Epoch 075/100 | Train: 0.3848 | Val: 0.3746 | LR: 1.93e-04 | Time: 1025.2s
      ✓ Val 